# 03 — Analysis & Cluster Maps

Load a pre-built feature matrix, cluster regions, and render geo-mapped results.

**Prerequisites:** Run `gtfs-features --scale agglomeration` to produce the feature matrix.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from swiss_gtfs.analysis.cluster import cluster_regions, cluster_to_dataframe
from swiss_gtfs.analysis.geo_join import merge_clusters, plot_cluster_map
from swiss_gtfs.mappings.regions import get_mapping

In [ ]:
# --- Configuration ---
SCALE         = 'agglomeration'
FEATURE_METHOD = 'stats'     # 'stats' or 'landscape'
CLUSTER_METHOD = 'kmeans'    # 'kmeans', 'agglomerative', 'dbscan'
N_CLUSTERS    = 5
FEATURES_DIR  = '../outputs/features'
GEODATA_DIR   = '../geodata/'

In [ ]:
# Load feature matrix
npz_path = f'{FEATURES_DIR}/{SCALE}/feature_matrix_{FEATURE_METHOD}.npz'
data = np.load(npz_path, allow_pickle=True)
city_keys = list(data['keys'])
matrix = data['matrix']
print(f'Feature matrix: {matrix.shape}')

In [ ]:
# Cluster
labels = cluster_regions(matrix, method=CLUSTER_METHOD, n_clusters=N_CLUSTERS)
results_df = cluster_to_dataframe(city_keys, labels)

print(results_df['cluster'].value_counts())

In [ ]:
# Add human-readable shapefile names for display
mapping = get_mapping(SCALE)
results_df['shapefile_name'] = results_df['city_key'].map(mapping)
results_df.head(10)

In [ ]:
# Geo-join and render choropleth map
merged = merge_clusters(results_df, SCALE, GEODATA_DIR)
m = plot_cluster_map(merged, SCALE)
m

In [ ]:
# UMAP embedding (optional — requires umap-learn)
try:
    import umap
    from sklearn.preprocessing import StandardScaler

    scaled = StandardScaler().fit_transform(matrix)
    reducer = umap.UMAP(n_components=2, random_state=42)
    embedding = reducer.fit_transform(scaled)

    results_df['umap_x'] = embedding[:, 0]
    results_df['umap_y'] = embedding[:, 1]
    results_df['cluster_str'] = results_df['cluster'].astype(str)

    import seaborn as sns
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=results_df, x='umap_x', y='umap_y',
                    hue='cluster_str', palette='Set1', s=60)
    for _, row in results_df.iterrows():
        plt.annotate(row['city_key'], (row['umap_x'], row['umap_y']),
                     fontsize=6, alpha=0.7)
    plt.title(f'UMAP — {SCALE} ({CLUSTER_METHOD}, k={N_CLUSTERS})')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('umap-learn not installed. Skipping UMAP plot.')

In [ ]:
# DBSCAN alternative
dbscan_labels = cluster_regions(matrix, method='dbscan', eps=0.8, min_samples=3)
dbscan_df = cluster_to_dataframe(city_keys, dbscan_labels, label_col='dbscan_cluster')
print(dbscan_df['dbscan_cluster'].value_counts())

dbscan_df['shapefile_name'] = dbscan_df['city_key'].map(mapping)
dbscan_merged = merge_clusters(dbscan_df, SCALE, GEODATA_DIR,
                               label_col='dbscan_cluster')
plot_cluster_map(dbscan_merged, SCALE, label_col='dbscan_cluster')